In [ ]:
peg_price_data = pd.read_csv(Path('Panel_A_Data') / 'Secondary_Prices_and_Peg_Deviations_Daily.csv')
redemption_pressure_data = pd.read_csv(Path('Panel_A_Data') / 'Redemption_Pressure_Index_Daily.csv')
SOFR_ = pd.read_csv(Path('Panel_B_Data') / 'SOFR.csv')
SOFR_OIS_1m = pd.read_csv(Path('Panel_B_Data') / 'USD_1M_SOFR_OIS.csv')
SOFR_OIS_3m = pd.read_csv(Path('Panel_B_Data') / 'USD_3M_SOFR_OIS.csv')

# Equation A1: peg shortfall (bps) = max[0, (1 - price in USD) x 10,000]
# Equation A2: redemption pressure (bps) = peg shortfall (bps) + net burn intensity (bps)
# Net burn intensity in the source file was constructed as:
# max(0, -daily supply change) / previous-day circulating supply x 10,000.
constructed_redemption_pressure = redemption_pressure_data[['Date']].copy()
for symbol in ['USDT', 'USDC', 'DAI']:
    price = peg_price_data[f'{symbol}_Price_USD']
    peg_shortfall = np.maximum(0.0, (1.0 - price) * 10_000)
    burn_intensity = redemption_pressure_data[f'{symbol}_Net_Burn_Intensity_bps']
    constructed_redemption_pressure[f'{symbol}_Peg_Shortfall_bps_eq_A1'] = peg_shortfall
    constructed_redemption_pressure[f'{symbol}_Redemption_Pressure_bps_eq_A2'] = (
        peg_shortfall + burn_intensity
    )

# Equations B1/B2: SOFR-OIS spread (bps) = (SOFR rate - OIS rate) x 100
sofr_for_equations = SOFR_.rename(columns={'observation_date': 'Date'}).copy()
ois_1m_for_equations = SOFR_OIS_1m.copy()
ois_3m_for_equations = SOFR_OIS_3m.copy()
for frame in [sofr_for_equations, ois_1m_for_equations, ois_3m_for_equations]:
    frame['Date'] = pd.to_datetime(frame['Date'], utc=True).dt.tz_localize(None).dt.normalize()

constructed_sofr_ois_spreads = (sofr_for_equations[['Date', 'SOFR']].merge(ois_1m_for_equations[['Date', 'USD_1M_SOFR_OIS']], on='Date', how='inner').merge(ois_3m_for_equations[['Date', 'USD_3M_SOFR_OIS']], on='Date', how='inner').sort_values('Date').reset_index(drop=True))
constructed_sofr_ois_spreads['SOFR_minus_1M_OIS_bps_eq_B1'] = (constructed_sofr_ois_spreads['SOFR'].sub(constructed_sofr_ois_spreads['USD_1M_SOFR_OIS']).mul(100))

constructed_sofr_ois_spreads['SOFR_minus_3M_OIS_bps_eq_B2'] = (
    constructed_sofr_ois_spreads['SOFR']
    .sub(constructed_sofr_ois_spreads['USD_3M_SOFR_OIS'])
    .mul(100)
)

# Saved DataFrame variables:
# constructed_redemption_pressure (Equations A1-A2)
# constructed_sofr_ois_spreads (Equations B1-B2)

In [ ]:
# Extract Circle's monthly USDC reserve-composition reports
# Note: Circle has no public historical reserve-composition API. Its official
# transparency page links the monthly assurance PDFs used below.
import re
from io import BytesIO
from urllib.parse import urljoin, unquote
import requests
from bs4 import BeautifulSoup
import pdfplumber

CIRCLE_TRANSPARENCY_URL = 'https://www.circle.com/transparency'
CIRCLE_HEADERS = {'User-Agent': 'Risk-Transmission-Research/1.0'}

response = requests.get(CIRCLE_TRANSPARENCY_URL, headers=CIRCLE_HEADERS, timeout=60)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
report_urls = pd.Series([
    urljoin(CIRCLE_TRANSPARENCY_URL, link['href'])
    for link in soup.select('a[href]')
    if unquote(link['href']).lower().endswith('.pdf')
    and 'usdc' in unquote(link['href']).lower()
], dtype='string').drop_duplicates().tolist()

if not report_urls:
    raise RuntimeError('No USDC monthly report links were returned by Circle.')

month_numbers = {
    month.lower(): number for number, month in enumerate(
        ['January', 'February', 'March', 'April', 'May', 'June',
         'July', 'August', 'September', 'October', 'November', 'December'], 1
    )
}

def month_from_report_url(url):
    decoded = unquote(url)
    year = re.search(r'(?<!\d)(20\d{2})(?!\d)', decoded)
    month = re.search('|'.join(month_numbers), decoded, flags=re.IGNORECASE)
    if year and month:
        return pd.Timestamp(int(year.group()), month_numbers[month.group().lower()], 1)
    return pd.NaT

circle_monthly_report_catalog = pd.DataFrame({'Report_URL': report_urls})
circle_monthly_report_catalog['Report_Month'] = (
    circle_monthly_report_catalog['Report_URL'].map(month_from_report_url)
)
circle_monthly_report_catalog = (
    circle_monthly_report_catalog.dropna(subset=['Report_Month'])
    .sort_values('Report_Month').reset_index(drop=True)
)

# Extract every table row because Circle's PDF layout and category names vary by year.
reserve_rows = []
for report in circle_monthly_report_catalog.itertuples(index=False):
    pdf_response = requests.get(report.Report_URL, headers=CIRCLE_HEADERS, timeout=120)
    pdf_response.raise_for_status()
    with pdfplumber.open(BytesIO(pdf_response.content)) as pdf:
        for page_number, page in enumerate(pdf.pages, 1):
            for table_number, table in enumerate(page.extract_tables(), 1):
                for row_number, cells in enumerate(table, 1):
                    cleaned = [
                        re.sub(r'\s+', ' ', str(cell)).strip() if cell is not None else pd.NA
                        for cell in cells
                    ]
                    reserve_rows.append({
                        'Report_Month': report.Report_Month, 'Page': page_number,
                        'Table': table_number, 'Row': row_number,
                        'Cells': cleaned, 'Report_URL': report.Report_URL,
                    })

circle_monthly_reserve_composition_raw = pd.DataFrame(reserve_rows)
if circle_monthly_reserve_composition_raw.empty:
    raise RuntimeError('Reports downloaded, but no PDF tables were extracted.')

max_cells = circle_monthly_reserve_composition_raw['Cells'].map(len).max()
cell_data = pd.DataFrame(
    circle_monthly_reserve_composition_raw.pop('Cells').tolist(),
    columns=[f'Cell_{i}' for i in range(1, max_cells + 1)],
)
circle_monthly_reserve_composition = pd.concat(
    [circle_monthly_reserve_composition_raw, cell_data], axis=1
)
circle_monthly_reserve_composition.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Reserve_Composition_Raw.csv', index=False
)
circle_monthly_report_catalog.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Report_Catalog.csv', index=False
)
circle_monthly_reserve_composition.head()

In [ ]:
# Normalize the two existing Circle datasets into one monthly reserve dataset
# Undisclosed categories remain NaN; they are not treated as zero.
import re
import numpy as np
import pandas as pd

circle_catalog = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Report_Catalog.csv',
    parse_dates=['Report_Month'],
)
circle_raw = pd.read_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Reserve_Composition_Raw.csv',
    parse_dates=['Report_Month'],
    dtype={column: 'string' for column in ['Cell_1', 'Cell_2', 'Cell_3', 'Cell_4']},
)

cell_columns = [column for column in circle_raw.columns if column.startswith('Cell_')]
circle_raw['Row_Text'] = circle_raw[cell_columns].fillna('').agg(' | '.join, axis=1)
circle_raw['Label'] = circle_raw['Cell_1'].fillna('').str.lower().str.replace(r'\s+', ' ', regex=True)

def circle_amount(value):
    """Convert Circle PDF values such as '$22.2bn' or '(21,598,084)' to USD."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower().replace(',', '').replace('$', '')
    match = re.search(r'\(?-?\d+(?:\.\d+)?\)?', text)
    if not match:
        return np.nan
    token = match.group()
    negative = token.startswith('(') or token.startswith('-')
    number = float(token.strip('()-'))
    multiplier = 1e9 if re.search(r'\b(?:bn|billion)\b', text) else 1e6 if re.search(r'\b(?:mn|million)\b', text) else 1
    return -number * multiplier if negative else number * multiplier

def last_row_amount(row):
    """Use the last reported value when a report contains two observation dates."""
    values = [circle_amount(row[column]) for column in cell_columns[1:]]
    values = [value for value in values if pd.notna(value)]
    return values[-1] if values else np.nan

months = circle_catalog[['Report_Month', 'Report_URL']].drop_duplicates('Report_Month')
circle_monthly_reserves = months.copy()
for column in [
    'USDC_in_Circulation_USD', 'Total_Reserves_USD', 'Bank_Deposits_USD',
    'Treasury_Bills_USD', 'Reverse_Repo_USD',
]:
    circle_monthly_reserves[column] = np.nan

# Total reserves and circulation: where two dates are shown, retain the month-end value.
for pattern, output_column in [
    (r'usdc.*(?:in circulation|issued and outstanding)', 'USDC_in_Circulation_USD'),
    (r'fair value of assets.*(?:reserve|held)', 'Total_Reserves_USD'),
]:
    selected = circle_raw[circle_raw['Label'].str.contains(pattern, case=False, regex=True, na=False)].copy()
    selected['Amount'] = selected.apply(last_row_amount, axis=1)
    selected = selected.dropna(subset=['Amount']).groupby('Report_Month', as_index=False).tail(1)
    mapping = selected.set_index('Report_Month')['Amount']
    circle_monthly_reserves[output_column] = circle_monthly_reserves['Report_Month'].map(mapping)

# 2021 reports disclose asset allocations directly in billions and percentages.
allocation_rows = circle_raw[
    circle_raw['Label'].str.contains('cash.*equivalent|us treasur', regex=True, na=False)
    & circle_raw['Cell_2'].notna()
].copy()
allocation_rows['Amount'] = allocation_rows['Cell_2'].map(circle_amount) * 1e9
allocation_rows['Share'] = pd.to_numeric(
    allocation_rows['Cell_3'].str.replace('%', '', regex=False), errors='coerce'
) / 100

cash_2021 = allocation_rows[allocation_rows['Label'].str.contains('cash.*equivalent', regex=True)]
treasury_2021 = allocation_rows[allocation_rows['Label'].str.contains('us treasur', regex=True)]
cash_amounts = cash_2021.drop_duplicates('Report_Month', keep='last').set_index('Report_Month')['Amount']
treasury_amounts_2021 = treasury_2021.drop_duplicates('Report_Month', keep='last').set_index('Report_Month')['Amount']
circle_monthly_reserves['Bank_Deposits_USD'] = circle_monthly_reserves['Report_Month'].map(cash_amounts)
circle_monthly_reserves['Treasury_Bills_USD'] = circle_monthly_reserves['Report_Month'].map(treasury_amounts_2021)

# Later reports list total U.S. Treasury securities. Use the final row in each
# monthly report, which corresponds to the later/month-end observation date.
treasury_rows = circle_raw[
    circle_raw['Label'].str.replace(' ', '', regex=False).str.contains('totalu.s.treasurysecurities', regex=False, na=False)
].copy()
treasury_rows['Amount'] = treasury_rows.apply(last_row_amount, axis=1)
treasury_rows = treasury_rows.dropna(subset=['Amount']).groupby('Report_Month', as_index=False).tail(1)
treasury_amounts = treasury_rows.set_index('Report_Month')['Amount']
circle_monthly_reserves['Treasury_Bills_USD'] = (
    circle_monthly_reserves['Treasury_Bills_USD']
    .fillna(circle_monthly_reserves['Report_Month'].map(treasury_amounts))
)

# When total reserves and Treasuries are disclosed, the residual is reported as
# bank deposits/cash. Circle's reports do not separately disclose reverse repos.
residual_cash = circle_monthly_reserves['Total_Reserves_USD'] - circle_monthly_reserves['Treasury_Bills_USD']
circle_monthly_reserves['Bank_Deposits_USD'] = circle_monthly_reserves['Bank_Deposits_USD'].fillna(residual_cash)

for asset in ['Bank_Deposits', 'Treasury_Bills', 'Reverse_Repo']:
    circle_monthly_reserves[f'{asset}_Share'] = (
        circle_monthly_reserves[f'{asset}_USD'] / circle_monthly_reserves['Total_Reserves_USD']
    )

circle_monthly_reserves['Disclosure_Quality'] = np.select(
    [
        circle_monthly_reserves[['Bank_Deposits_USD', 'Treasury_Bills_USD']].notna().all(axis=1),
        circle_monthly_reserves['Total_Reserves_USD'].notna(),
    ],
    ['composition_available', 'total_only'],
    default='not_machine_readable',
)
circle_monthly_reserves = circle_monthly_reserves.sort_values('Report_Month').reset_index(drop=True)
circle_monthly_reserves.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Reserve_Composition_Normalized.csv',
    index=False,
)
circle_monthly_reserves.tail(12)

In [ ]:
# Tether reserve composition: official JSON endpoints + quarterly reports
import json
import re
from io import BytesIO
import numpy as np
import pandas as pd
import pdfplumber
import requests

TETHER_API_URL = 'https://app.tether.to/transparency.json'
TETHER_PAGE_DATA_URL = 'https://tether.to/page-data/transparency/page-data.json'
TETHER_HEADERS = {'User-Agent': 'Risk-Transmission-Research/1.0'}

# API 1: current token circulation and reserve balances by blockchain.
api_response = requests.get(TETHER_API_URL, headers=TETHER_HEADERS, timeout=60)
api_response.raise_for_status()
tether_api_raw = api_response.json()
tether_current_transparency = pd.json_normalize(tether_api_raw['data'], sep='__').T
tether_current_transparency = tether_current_transparency.reset_index()
tether_current_transparency.columns = ['Variable', 'Value']
tether_current_transparency.to_csv(
    PANEL_A_DATA_FOLDER / 'Tether_Transparency_API_Current.csv', index=False
)

# API 2: current reserve composition and official quarterly-report catalogue.
page_response = requests.get(TETHER_PAGE_DATA_URL, headers=TETHER_HEADERS, timeout=60)
page_response.raise_for_status()
page_data = page_response.json()['result']['data']

report_nodes = page_data['reports']['nodes']
tether_quarterly_report_catalog = pd.DataFrame({
    'Report_Date': pd.to_datetime([report['reportDate'] for report in report_nodes]),
    'Report_Name': [report['reportName'] for report in report_nodes],
    'Report_URL': [
        'https:' + report['reportFile']['file']['url']
        if report['reportFile']['file']['url'].startswith('//')
        else report['reportFile']['file']['url']
        for report in report_nodes
    ],
    'Page_Updated_At': [report.get('updatedAt') for report in report_nodes],
}).sort_values('Report_Date').reset_index(drop=True)
tether_quarterly_report_catalog.to_csv(
    PANEL_A_DATA_FOLDER / 'Tether_Quarterly_Report_Catalog.csv', index=False
)

# Current composition directly from Tether's page-data JSON.
current_date = tether_quarterly_report_catalog['Report_Date'].max()
current_nodes = page_data['enBreakdown']['nodes']
tether_current_reserve_composition = pd.DataFrame({
    'Report_Date': current_date,
    'Asset_Category': [item['title'] for item in current_nodes],
    'Parent_Category': [
        item.get('contentfulparent', {}).get('title')
        if item.get('contentfulparent') else pd.NA
        for item in current_nodes
    ],
    'Amount_USD': [item['value'] for item in current_nodes],
})
top_level_total = tether_current_reserve_composition.loc[
    tether_current_reserve_composition['Parent_Category'].isna(), 'Amount_USD'
].sum()
tether_current_reserve_composition['Share_of_Total_Reserves'] = (
    tether_current_reserve_composition['Amount_USD'] / top_level_total
)
tether_current_reserve_composition.to_csv(
    PANEL_A_DATA_FOLDER / 'Tether_Reserve_Composition_Current.csv', index=False
)

# Historical composition is published quarterly in the official PDFs.
asset_labels = [
    'U.S. Treasury Bills', 'Overnight Reverse Repurchase Agreements',
    'Term Reverse Repurchase Agreements', 'Money Market Funds',
    'Cash & Bank Deposits', 'Non-U.S. Treasury Bills', 'Commercial Paper',
    'Certificates of Deposit', 'Corporate Bonds', 'Funds',
    'Precious Metals', 'Bitcoin', 'Bitcoins', 'Public Equities',
    'Other Investments', 'Secured Loans',
]

def amount_after_label(text, label):
    """Return the first large USD number following an asset label."""
    match = re.search(re.escape(label), text, flags=re.IGNORECASE)
    if not match:
        return np.nan
    window = text[match.end():match.end() + 220]
    candidates = re.findall(r'(?<![\d.])\$?\s*([0-9]{1,3}(?:[ ,][0-9]{3})+|[0-9]{6,})(?!\d)', window)
    for candidate in candidates:
        # Some PDFs attach a footnote number before a space-separated amount,
        # for example '2 117,035,732,050'. Remove that footnote prefix.
        candidate = re.sub(r'^\d{1,2}\s+(?=\d{1,3},)', '', candidate)
        value = float(candidate.replace(',', '').replace(' ', ''))
        if value >= 100_000:
            return value
    return np.nan

historical_rows = []
for report in tether_quarterly_report_catalog.itertuples(index=False):
    if report.Report_Date < pd.Timestamp('2020-01-01'):
        continue
    pdf_response = requests.get(report.Report_URL, headers=TETHER_HEADERS, timeout=120)
    pdf_response.raise_for_status()
    with pdfplumber.open(BytesIO(pdf_response.content)) as pdf:
        report_text = '\n'.join((page.extract_text() or '') for page in pdf.pages)
    report_text = re.sub(r'[\t\r]+', ' ', report_text)
    report_text = re.sub(r' +', ' ', report_text)
    for label in asset_labels:
        amount = amount_after_label(report_text, label)
        if pd.notna(amount):
            historical_rows.append({
                'Report_Date': report.Report_Date,
                'Asset_Category': 'Bitcoin' if label == 'Bitcoins' else label,
                'Amount_USD': amount,
                'Report_URL': report.Report_URL,
            })

tether_quarterly_reserve_composition = pd.DataFrame(historical_rows)
tether_quarterly_reserve_composition = (
    tether_quarterly_reserve_composition
    .sort_values(['Report_Date', 'Asset_Category'])
    .drop_duplicates(['Report_Date', 'Asset_Category'], keep='last')
    .reset_index(drop=True)
)
quarterly_totals = tether_quarterly_reserve_composition.groupby('Report_Date')['Amount_USD'].transform('sum')
tether_quarterly_reserve_composition['Share_of_Extracted_Reserves'] = (
    tether_quarterly_reserve_composition['Amount_USD'] / quarterly_totals
)
tether_quarterly_reserve_composition.to_csv(
    PANEL_A_DATA_FOLDER / 'Tether_Reserve_Composition_Quarterly.csv', index=False
)

# Explicit quarterly coverage from 2020. Tether published no 2020 reserve
# composition reports, so those quarters are retained and marked unavailable.
tether_quarterly_coverage = pd.DataFrame({
    'Report_Date': pd.date_range('2020-03-31', current_date, freq='QE'),
})
available_dates = set(tether_quarterly_reserve_composition['Report_Date'])
catalog_dates = set(tether_quarterly_report_catalog['Report_Date'])
tether_quarterly_coverage['Official_Report_Available'] = (
    tether_quarterly_coverage['Report_Date'].isin(catalog_dates)
)
tether_quarterly_coverage['Composition_Available'] = (
    tether_quarterly_coverage['Report_Date'].isin(available_dates)
)
tether_quarterly_coverage['Disclosure_Note'] = np.select(
    [
        tether_quarterly_coverage['Composition_Available'],
        tether_quarterly_coverage['Official_Report_Available'],
    ],
    ['category composition extracted', 'official report without extractable category composition'],
    default='Tether did not publish a quarterly reserve-composition report',
)
tether_quarterly_coverage.to_csv(
    PANEL_A_DATA_FOLDER / 'Tether_Reserve_Composition_Coverage_From_2020.csv', index=False
)

tether_quarterly_reserve_composition.tail(20)

In [ ]:
# Circle monthly reserve composition from official PDFs (May 2021 onward)
import re
from io import BytesIO
from urllib.parse import urljoin, unquote
import numpy as np
import requests
import pdfplumber
from bs4 import BeautifulSoup

CIRCLE_URL = 'https://www.circle.com/transparency'
HEADERS = {'User-Agent': 'Risk-Transmission-Research/1.0'}
html = requests.get(CIRCLE_URL, headers=HEADERS, timeout=60)
html.raise_for_status()
soup = BeautifulSoup(html.text, 'html.parser')
months = {m.lower(): i for i, m in enumerate(['January','February','March','April','May','June','July','August','September','October','November','December'], 1)}

def report_month(url):
    text = unquote(url)
    year = re.search(r'(?<!\d)(20\d{2})(?!\d)', text)
    month = re.search('|'.join(months), text, re.I)
    return pd.Timestamp(int(year.group()), months[month.group().lower()], 1) if year and month else pd.NaT

urls = pd.Series([urljoin(CIRCLE_URL, a['href']) for a in soup.select('a[href]')
                  if unquote(a['href']).lower().endswith('.pdf') and 'usdc' in unquote(a['href']).lower()]).drop_duplicates()
circle_report_catalog = pd.DataFrame({'Report_URL': urls})
circle_report_catalog['Report_Month'] = circle_report_catalog['Report_URL'].map(report_month)
circle_report_catalog = circle_report_catalog[circle_report_catalog.Report_Month >= '2021-05-01'].sort_values('Report_Month').drop_duplicates('Report_Month', keep='last')

patterns = {
 'Cash_and_Cash_Equivalents': r'^cash\s*(?:&|and)\s*cash\s*equivalents',
 'Yankee_CDs': r'^yankee\s+cds', 'US_Treasuries': r'^(?:total\s+)?u\.?s\.?\s+treasur',
 'Commercial_Paper': r'^commercial\s+paper', 'Corporate_Bonds': r'^corporate\s+bonds?',
 'Municipal_Bonds_and_US_Agencies': r'^municipal\s+bonds?',
 'Certificates_of_Deposit': r'^certificates?\s+of\s+deposit',
 'Money_Market_Funds': r'^money\s+market\s+funds?',
 'Total_Reserves': r'^(?:fair\s+value\s+of\s+assets.*reserve|total\s+reserves?)',
 'USDC_in_Circulation': r'^usdc.*(?:in\s+circulation|issued\s+and\s+outstanding)'}

def clean(x): return re.sub(r'\s+', ' ', str(x)).strip() if x is not None else ''
def number(x, billions=False):
    text = clean(x).lower().replace('$','').replace(',','')
    match = re.search(r'\(?-?\d+(?:\.\d+)?\)?', text)
    if not match: return np.nan
    token = match.group(); value = float(token.strip('()-'))
    if token.startswith('(') or token.startswith('-'): value = -value
    if billions or 'bn' in text or 'billion' in text: value *= 1e9
    elif 'mn' in text or 'million' in text: value *= 1e6
    return value

rows = []
for report in circle_report_catalog.itertuples(index=False):
    pdf_bytes = requests.get(report.Report_URL, headers=HEADERS, timeout=120)
    pdf_bytes.raise_for_status()
    with pdfplumber.open(BytesIO(pdf_bytes.content)) as pdf:
        for page_no, page in enumerate(pdf.pages, 1):
            for table_no, table in enumerate(page.extract_tables(), 1):
                uses_billions = bool(re.search(r'\$?\s*bn\b', ' '.join(clean(c) for r in table for c in r), re.I))
                for raw_row in table:
                    cells = [clean(c) for c in raw_row]
                    if not cells or not cells[0]: continue
                    label = re.sub(r'(?<=[A-Za-z])\d+$', '', cells[0]).strip()
                    category = next((name for name, pattern in patterns.items() if re.search(pattern, label, re.I)), None)
                    if not category: continue
                    shares = [number(c)/100 for c in cells[1:] if '%' in c]
                    amounts = [number(c, uses_billions) for c in cells[1:] if c and '%' not in c]
                    amounts = [v for v in amounts if pd.notna(v) and abs(v) >= 100000]
                    if amounts or shares:
                        rows.append({'Report_Month':report.Report_Month,'Reserve_Category':category,
                          'Amount_USD':amounts[-1] if amounts else np.nan,
                          'Allocation_Share':shares[-1] if shares else np.nan,
                          'Original_Label':label,'Page':page_no,'Table':table_no,'Report_URL':report.Report_URL})

Circle_USDC_monthly_reserve_composition = pd.DataFrame(rows).sort_values(['Report_Month','Page','Table']).drop_duplicates(['Report_Month','Reserve_Category'], keep='last').reset_index(drop=True)
Circle_USDC_monthly_reserve_composition['Extraction_Method'] = 'official_pdf_table'
Circle_USDC_monthly_reserve_composition.to_csv(PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Reserve_Composition_From_May_2021.csv', index=False)
non_asset_rows = ['Total_Reserves', 'USDC_in_Circulation']
Circle_USDC_monthly_asset_composition = Circle_USDC_monthly_reserve_composition[
    ~Circle_USDC_monthly_reserve_composition['Reserve_Category'].isin(non_asset_rows)
].copy()
Circle_USDC_monthly_asset_composition.loc[
    Circle_USDC_monthly_asset_composition['Allocation_Share'].eq(0)
    & Circle_USDC_monthly_asset_composition['Amount_USD'].isna(),
    'Amount_USD',
] = 0
Circle_USDC_monthly_asset_composition.to_csv(
    PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_Asset_Composition_Disclosed.csv', index=False
)
circle_report_catalog.to_csv(PANEL_A_DATA_FOLDER / 'Circle_USDC_Monthly_PDF_Catalog_From_May_2021.csv', index=False)
Circle_USDC_monthly_asset_composition

In [ ]:
# Daily 5-year bank CDS, exposed-bank equities and KBW Bank Index (2020-latest)
import re
import numpy as np
import pandas as pd
import lseg.data as ld

START_DATE = '2020-01-01'
END_DATE = pd.Timestamp.today().normalize().strftime('%Y-%m-%d')
PANEL_B_DATA_FOLDER.mkdir(parents=True, exist_ok=True)

# Major US custodial banks and banks exposed to stablecoin reserve/funding flows.
banks = {
    'BNY_Mellon': {'search_name': 'Bank of New York Mellon', 'equity_ric': 'BK.N'},
    'State_Street': {'search_name': 'State Street Corporation', 'equity_ric': 'STT.N'},
    'JPMorgan': {'search_name': 'JPMorgan Chase', 'equity_ric': 'JPM.N'},
    'Bank_of_America': {'search_name': 'Bank of America', 'equity_ric': 'BAC.N'},
    'Citigroup': {'search_name': 'Citigroup', 'equity_ric': 'C.N'},
}

def flatten_history(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = ['_'.join(str(x) for x in column if str(x) != '') for column in frame.columns]
    return frame.rename_axis('Date').reset_index()

ld.open_session()
try:
    # Discover CDS RICs because CDS identifiers can vary by contributor.
    discovery_frames = []
    selected_cds = {}
    for bank, identifiers in banks.items():
        found = ld.discovery.search(
            query=identifiers['search_name'],
            view=ld.discovery.Views.CDS_QUOTES,
            top=100,
        ).copy()
        found.insert(0, 'Bank', bank)
        discovery_frames.append(found)

        searchable = found.astype('string').fillna('').agg(' | '.join, axis=1)
        five_year = searchable.str.contains(r'(?i)(?:5\s*Y|5\s*YR|5\s*YEAR)')
        senior = searchable.str.contains(r'(?i)(?:SENIOR|SNR|SR)')
        usd = searchable.str.contains(r'(?i)(?:USD|US DOLLAR)')
        candidates = found.loc[five_year & senior & usd].copy()
        if candidates.empty:
            candidates = found.loc[five_year].copy()

        ric_column = next((column for column in candidates.columns if str(column).upper() == 'RIC'), None)
        if ric_column is None or candidates.empty:
            raise RuntimeError(f'No unambiguous 5-year CDS RIC found for {bank}; inspect the discovery CSV.')
        candidates['_Composite_Priority'] = candidates[ric_column].astype('string').str.endswith('=R').astype(int)
        candidates = candidates.sort_values('_Composite_Priority', ascending=False)
        selected_cds[bank] = candidates.iloc[0][ric_column]

    CDS_discovery_results = pd.concat(discovery_frames, ignore_index=True)
    CDS_discovery_results.to_csv(
        PANEL_B_DATA_FOLDER / 'Bank_5Y_CDS_RIC_Discovery.csv', index=False
    )
    pd.DataFrame([
        {'Bank': bank, 'CDS_RIC': ric} for bank, ric in selected_cds.items()
    ]).to_csv(PANEL_B_DATA_FOLDER / 'Bank_5Y_CDS_Selected_RICs.csv', index=False)

    # CDS historical fields are spreads in basis points. PAR_MID1 is the daily
    # par-spread close; MID_O/H/L_SPD retain the daily spread range.
    Bank_5Y_CDS_Daily = flatten_history(ld.get_history(
        universe=list(selected_cds.values()),
        fields=['MID_O_SPD', 'MID_H_SPD', 'MID_L_SPD', 'PAR_MID1'],
        start=START_DATE, end=END_DATE, interval='daily',
    ))
    for bank, ric in selected_cds.items():
        par_column = f'{ric}_PAR_MID1'
        if par_column in Bank_5Y_CDS_Daily:
            Bank_5Y_CDS_Daily[f'{bank}_5Y_CDS_bps'] = Bank_5Y_CDS_Daily[par_column]
    cds_availability = pd.DataFrame({
        'Bank': list(selected_cds),
        'CDS_RIC': list(selected_cds.values()),
        'Usable_History': [
            Bank_5Y_CDS_Daily.get(f'{bank}_5Y_CDS_bps', pd.Series(dtype=float)).notna().any()
            for bank in selected_cds
        ],
    })
    cds_availability.to_csv(PANEL_B_DATA_FOLDER / 'Bank_5Y_CDS_Availability.csv', index=False)
    Bank_5Y_CDS_Daily = Bank_5Y_CDS_Daily.dropna(axis=1, how='all')
    Bank_5Y_CDS_Daily.to_csv(PANEL_B_DATA_FOLDER / 'Bank_5Y_CDS_Daily.csv', index=False)

    # Adjusted daily equity close for the exposed/custodial-bank basket.
    equity_rics = [identifiers['equity_ric'] for identifiers in banks.values()]
    Exposed_Bank_Equities_Daily = flatten_history(ld.get_history(
        universe=equity_rics, fields=['TR.ClosePrice(Adjusted=1)'],
        start=START_DATE, end=END_DATE, interval='daily',
    ))
    Exposed_Bank_Equities_Daily.to_csv(
        PANEL_B_DATA_FOLDER / 'Exposed_Custodial_Bank_Equities_Daily.csv', index=False
    )

    # KBW Nasdaq Bank Index.
    KBW_Bank_Index_Daily = flatten_history(ld.get_history(
        universe=['.BKX'], fields=['TRDPRC_1'],
        start=START_DATE, end=END_DATE, interval='daily',
    ))
    KBW_Bank_Index_Daily.to_csv(
        PANEL_B_DATA_FOLDER / 'KBW_Bank_Index_Daily.csv', index=False
    )
finally:
    ld.close_session()

Bank_5Y_CDS_Daily.tail(), Exposed_Bank_Equities_Daily.tail(), KBW_Bank_Index_Daily.tail()

In [ ]:
# Panel E: euro-area short bills and matched non-reserve short-duration assets
import numpy as np
import pandas as pd
import lseg.data as ld

PANEL_E_DATA_FOLDER = Path('Panel_E_Data')
PANEL_E_DATA_FOLDER.mkdir(parents=True, exist_ok=True)
START_DATE = '2020-01-01'
END_DATE = pd.Timestamp.today().normalize().strftime('%Y-%m-%d')

# German and French sovereign benchmarks proxy euro-area 1M/3M bill yields.
# UK 1M/3M bills provide maturity-matched sovereign controls not reported in
# the Circle or Tether reserve-composition datasets used in this project.
bill_rics = {
    'Germany_1M_Bill_Yield': 'DE1MT=RR',
    'Germany_3M_Bill_Yield': 'DE3MT=RR',
    'France_1M_Bill_Yield': 'FR1MT=RR',
    'France_3M_Bill_Yield': 'FR3MT=RR',
    'UK_1M_Bill_Yield': 'GB1MT=RR',
    'UK_3M_Bill_Yield': 'GB3MT=RR',
}

def flatten_lseg_history(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [
            '_'.join(str(value) for value in column if str(value) != '')
            for column in frame.columns
        ]
    return frame.rename_axis('Date').reset_index()

ld.open_session()
try:
    bill_history_raw = flatten_lseg_history(ld.get_history(
        universe=list(bill_rics.values()),
        fields=['MID_YLD_1', 'YLDTOMAT', 'B_YLD_1', 'A_YLD_1'],
        start=START_DATE,
        end=END_DATE,
        interval='daily',
    ))
finally:
    ld.close_session()

# Construct one daily yield per RIC. Prefer LSEG's mid yield; otherwise use
# yield-to-maturity, then the midpoint of bid and ask yields.
Placebo_Short_Duration_Assets_Daily = bill_history_raw[['Date']].copy()
for output_name, ric in bill_rics.items():
    mid_column = f'{ric}_MID_YLD_1'
    ytm_column = f'{ric}_YLDTOMAT'
    bid_column = f'{ric}_B_YLD_1'
    ask_column = f'{ric}_A_YLD_1'

    series = pd.Series(np.nan, index=bill_history_raw.index, dtype='float64')
    if mid_column in bill_history_raw:
        series = pd.to_numeric(bill_history_raw[mid_column], errors='coerce')
    if ytm_column in bill_history_raw:
        series = series.fillna(pd.to_numeric(bill_history_raw[ytm_column], errors='coerce'))
    if bid_column in bill_history_raw and ask_column in bill_history_raw:
        bid_ask_mid = bill_history_raw[[bid_column, ask_column]].apply(pd.to_numeric, errors='coerce').mean(axis=1)
        series = series.fillna(bid_ask_mid)
    Placebo_Short_Duration_Assets_Daily[output_name] = series

Euro_Area_Short_Bills_Daily = Placebo_Short_Duration_Assets_Daily[[
    'Date', 'Germany_1M_Bill_Yield', 'Germany_3M_Bill_Yield',
    'France_1M_Bill_Yield', 'France_3M_Bill_Yield',
]].copy()
Matched_NonReserve_Short_Bills_Daily = Placebo_Short_Duration_Assets_Daily[[
    'Date', 'UK_1M_Bill_Yield', 'UK_3M_Bill_Yield',
]].copy()

Euro_Area_Short_Bills_Daily.to_csv(
    PANEL_E_DATA_FOLDER / 'Euro_Area_Short_Bills_1M_3M_Daily.csv', index=False
)
Matched_NonReserve_Short_Bills_Daily.to_csv(
    PANEL_E_DATA_FOLDER / 'Matched_NonReserve_UK_Short_Bills_1M_3M_Daily.csv', index=False
)
Placebo_Short_Duration_Assets_Daily.to_csv(
    PANEL_E_DATA_FOLDER / 'Placebo_Short_Duration_Assets_Daily.csv', index=False
)
bill_history_raw.to_csv(
    PANEL_E_DATA_FOLDER / 'Placebo_Short_Duration_Assets_LSEG_Raw.csv', index=False
)

Euro_Area_Short_Bills_Daily.tail(), Matched_NonReserve_Short_Bills_Daily.tail()

In [ ]:
# Government MMF total AUM and monthly net flows from MMF_by_cata.csv
PANEL_B_DATA_FOLDER = Path('Panel_B_Data')
MMF_source = pd.read_csv(PANEL_B_DATA_FOLDER / 'MMF_by_cata.csv', skiprows=2)
MMF_source['Date'] = pd.to_datetime(MMF_source['Date'], format='%m/%d/%Y')

# The OFR classification changes over time, so total Government MMF AUM is
# the sum of the legacy Government column and the two fees/gates categories.
government_columns = [
    'Government (No Fees & Gates)',
    'Government',
    'Government (Fees & Gates)',
]
MMF_source[government_columns] = MMF_source[government_columns].apply(
    pd.to_numeric, errors='coerce'
).fillna(0)
MMF_source['Government_MMF_Total_AUM_USD'] = MMF_source[government_columns].sum(axis=1)

# Monthly net flow is constructed as the month-over-month change in total AUM.
# It is calculated before filtering so January 2020 uses December 2019 as its base.
MMF_source['Government_MMF_Net_Flow_USD'] = (
    MMF_source['Government_MMF_Total_AUM_USD'].diff()
)
MMF_2020_latest = MMF_source.loc[MMF_source['Date'] >= '2020-01-01'].copy()

Government_MMF_Total_AUM = MMF_2020_latest[[
    'Date', 'Government_MMF_Total_AUM_USD'
]].reset_index(drop=True)
Government_MMF_Net_Flows = MMF_2020_latest[[
    'Date', 'Government_MMF_Net_Flow_USD'
]].reset_index(drop=True)

Government_MMF_Total_AUM.to_csv(
    PANEL_B_DATA_FOLDER / 'Government_MMF_Total_AUM_Monthly.csv', index=False
)
Government_MMF_Net_Flows.to_csv(
    PANEL_B_DATA_FOLDER / 'Government_MMF_Net_Flows_Monthly.csv', index=False
)

Government_MMF_Total_AUM.tail(), Government_MMF_Net_Flows.tail()